# GNN-Based BERT for Understanding Context from Music
### End-to-End Inference Demo (`notebooks/demo_context.ipynb`)
**Course**: Neural Networks (CSE425 / EEE474 / CSE715)  
**Objective**: Demonstrate multi-modal forward pass combining audio structure graphs (GNN) and natural-language text descriptions (BERT) via Cross-Attention Fusion.

In [ ]:
import os
import sys
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import networkx as nx

# Add project src to path
sys.path.insert(0, os.path.abspath(".."))

from src.audio_features import generate_synthetic_music_track, segment_audio, extract_segment_features
from src.graph_builder import build_music_structure_graph
from src.gnn_model import MusicStructureGNN
from src.fusion_model import GNNBertFusionModel
from src.dataset import GENRE_TAGS

print("Libraries loaded successfully!")

## 1. Generate / Load Audio & Extract Segment Features
We load a 30-second audio track, slice it into 5-second segments, and extract pooled log-mel (128 bins) and chroma (12 bins) feature vectors for each segment.

In [ ]:
# Synthesize/load an audio clip with chord progressions (e.g. Rock track)
waveform, meta = generate_synthetic_music_track(duration=30.0, sr=22050, genre_seed=0)
print(f"Loaded Track: {meta['track_id']} | Ground-Truth Genre: {meta['genre']}")
print(f"Description: \"{meta['caption']}\"")

# Slice into 5-second segments
segments = segment_audio(waveform, segment_duration=5.0)
node_features = extract_segment_features(segments)
print(f"Extracted {len(segments)} segments. Node feature matrix shape: {node_features.shape}")

## 2. Construct Music Structure Graph G = (V, E)
Nodes represent temporal audio slices. Edges include:
- **Temporal sequential transitions** (adjacent segments)
- **Harmonic / Timbral similarity edges** (cosine similarity > 0.65)

In [ ]:
graph_dict = build_music_structure_graph(node_features, similarity_threshold=0.65, metadata=meta)
G_nx = graph_dict["networkx_graph"]

plt.figure(figsize=(7, 5))
pos = nx.spring_layout(G_nx, seed=42)
nx.draw_networkx_nodes(G_nx, pos, node_color='skyblue', node_size=600, edgecolors='black')
nx.draw_networkx_labels(G_nx, pos, font_size=11, font_weight='bold')

# Distinguish temporal vs similarity edges
temporal_edges = [(u, v) for u, v, d in G_nx.edges(data=True) if d.get('edge_type') == 'temporal']
similarity_edges = [(u, v) for u, v, d in G_nx.edges(data=True) if d.get('edge_type') == 'similarity']

nx.draw_networkx_edges(G_nx, pos, edgelist=temporal_edges, edge_color='gray', width=2, label='Temporal')
nx.draw_networkx_edges(G_nx, pos, edgelist=similarity_edges, edge_color='crimson', width=2, style='dashed', label='Similarity')

plt.title(f"Music Structure Graph ({meta['genre']})", fontsize=13)
plt.legend(loc='lower left')
plt.axis('off')
plt.show()

## 3. End-to-End Inference: GNN-BERT Cross-Attention Fusion
We pass the graph into GraphSAGE to get readout `g`, pass the caption into BERT to get `H_text`, and compute multi-modal predictions via Cross-Attention.

In [ ]:
# Initialize models
gnn = MusicStructureGNN(in_channels=140, hidden_channels=128, out_channels=128, num_classes=10)
fusion = GNNBertFusionModel(gnn_dim=128, text_dim=768, fusion_dim=128, num_classes=10, fusion_type="cross_attention")

# Prepare graph tensors
x_tensor = torch.tensor(graph_dict["node_features"], dtype=torch.float32)
edge_index = torch.tensor(graph_dict["edge_index"], dtype=torch.long).t().contiguous()

gnn.eval()
fusion.eval()
with torch.no_grad():
    # 1. GNN graph readout
    _, g = gnn(x_tensor, edge_index)
    
    # 2. Simulated BERT contextual embedding for prompt
    H_text = torch.randn(1, 32, 768)
    
    # 3. Cross-attention fusion forward pass
    tag_logits, emotion_preds, z, attn_map = fusion(g, H_text=H_text)
    tag_probs = torch.sigmoid(tag_logits).numpy()[0]
    valence, arousal = emotion_preds.numpy()[0]

print("=== INFERENCE RESULTS ===")
print(f"Predicted Valence: {valence:.2f} (Target: {meta['valence']:.2f})")
print(f"Predicted Arousal: {arousal:.2f} (Target: {meta['arousal']:.2f})")
print("\nTop Predicted Tags:")
top_indices = np.argsort(tag_probs)[::-1][:5]
for rank, idx in enumerate(top_indices):
    print(f"  {rank + 1}. {GENRE_TAGS[idx]:<12} Confidence: {tag_probs[idx]:.1%}")